In [13]:
import numpy as np

def hel_mat(NMax, k=4):
    """
    Return a matrix to change the basis of T-matrices.
    
    Parameters
    ----------
    NMax : int
        Maximum expansion order of the T-matrix.
    k : int, optional
        Position where the minus sign is (default = 4).
        
    Returns
    -------
    H : ndarray
        Basis change matrix.
    """
    d_half = NMax * (NMax + 2)
    I = np.eye(d_half)

    if k == 4:
        H = np.block([[I,  I], [I, -I]]) / np.sqrt(2)
    elif k == 3:
        H = np.block([[ I,  I], [-I,  I]]) / np.sqrt(2)
    elif k == 2:
        H = np.block([[ I, -I], [ I,  I]]) / np.sqrt(2)
    elif k == 1:
        H = np.block([[-I,  I], [ I,  I]]) / np.sqrt(2)
    else:
        raise ValueError("k must be 1, 2, 3, or 4")

    return H

import numpy as np

In [15]:
def cts(x, y, z):
    """
    Convert Cartesian to spherical coordinates.

    Parameters
    ----------
    x, y, z : float or ndarray
        Cartesian coordinates.

    Returns
    -------
    r : float or ndarray
        Radial distance.
    theta : float or ndarray
        Polar angle (0 <= theta <= pi).
    phi : float or ndarray
        Azimuthal angle (-pi < phi <= pi).
    """
    # Handle the special case (0,0,0)
    if np.isscalar(x) and np.isscalar(y) and np.isscalar(z):
        if x == 0 and y == 0 and z == 0:
            return 0.0, 0.0, 0.0

    r = np.sqrt(x**2 + y**2 + z**2)
    phi = np.arctan2(y, x)         # azimuth angle
    theta = np.arccos(np.divide(z, r, out=np.zeros_like(z, dtype=float), where=r!=0))  # polar angle

    return r, theta, phi

def sph_bessel(x, l):
    """
    Spherical Bessel function of the first kind j_l(x).

    Parameters
    ----------
    x : float or ndarray
        Argument.
    l : int
        Order.

    Returns
    -------
    res : float or ndarray
        Value of the spherical Bessel function j_l(x).
    """
    return scipy.special.spherical_jn(l, x)

def sph_hankel(x, l):
    """
    Spherical Hankel function of the first kind h_l^(1)(x).

    Parameters
    ----------
    x : float or ndarray
        Argument.
    l : int
        Order.

    Returns
    -------
    res : complex float or ndarray
        Value of h_l^(1)(x).
    """
    x = np.asarray(x, dtype=complex)  # allow complex arguments
    return scipy.special.hankel1(l + 0.5, x) * np.sqrt(np.pi / (2 * x))

def wignerD(j, *args):
    """
    Return the Wigner D-matrix or a single element.

    Reference:
        D. A. Varshalovich, A. N. Moskalev, and V. K. Khersonskii.
        Quantum Theory of Angular Momentum, World Scientific, 1988. p.77 (4.3.5)
    """
    if len(args) == 6:  # Single element
        m, n, alpha, beta, gamma, facvec = args
        if abs(beta) <= np.finfo(float).eps:
            return np.exp(-1j * (alpha + gamma) * m) if m == n else 0.0

        k_vals = np.arange(max(0, n - m), min(j + n, j - m) + 1)

        terms = ((-1)**k_vals *
                 np.cos(beta/2.0)**(2*j + n - m - 2*k_vals) *
                 np.sin(beta/2.0)**(2*k_vals + m - n) /
                 (facvec[j+n-k_vals] *
                  facvec[j-m-k_vals] *
                  facvec[k_vals+m-n] *
                  facvec[k_vals]))

        summation = np.sum(terms)

        prefactor = (np.exp(-1j*alpha*m) *
                     np.exp(-1j*gamma*n) *
                     np.sqrt(facvec[j+n] * facvec[j-n] *
                             facvec[j+m] * facvec[j-m]) *
                     (-1)**(m-n))

        return prefactor * summation

    elif len(args) == 4:  # Full matrix
        alpha, beta, gamma, facvec = args
        if abs(beta) <= np.finfo(float).eps:
            m_vals = np.arange(-j, j+1)
            return np.diag(np.exp(-1j * (alpha + gamma) * m_vals))

        D_j = np.zeros((2*j+1, 2*j+1), dtype=complex)
        for m in range(-j, j+1):
            for n in range(-j, j+1):
                k_vals = np.arange(max(0, n - m), min(j + n, j - m) + 1)

                terms = ((-1)**k_vals *
                         np.cos(beta/2.0)**(2*j + n - m - 2*k_vals) *
                         np.sin(beta/2.0)**(2*k_vals + m - n) /
                         (facvec[j+n-k_vals] *
                          facvec[j-m-k_vals] *
                          facvec[k_vals+m-n] *
                          facvec[k_vals]))

                summation = np.sum(terms)

                prefactor = (np.exp(-1j*alpha*m) *
                             np.exp(-1j*gamma*n) *
                             np.sqrt(facvec[j+n] * facvec[j-n] *
                                     facvec[j+m] * facvec[j-m]) *
                             (-1)**(m-n))

                D_j[m+j, n+j] = prefactor * summation

        return D_j

    else:
        raise ValueError("Invalid number of arguments for wignerD")

import warnings

def wigner3j(j1, j2, j3, m1, m2, m3, facvec):
    """
    Wigner 3j symbol via backward recursion (Xu 1998, Eqs. 6–8,11).

    Parameters
    ----------
    j1, j2 : int
    j3 : int or array-like of int
        j3 can be an array; the result will match its shape.
    m1, m2, m3 : int
        Must satisfy m1 + m2 + m3 = 0 for a non-zero result (selection rule).
    facvec : 1D ndarray
        Precomputed factorials with MATLAB-like indexing:
        facvec[k] == k! for k >= 0. (So facvec[0] = 0!, facvec[1] = 1!, ...)

    Returns
    -------
    a : ndarray
        Wigner 3j values for each j3, same shape as input j3.
    """
    j3_arr = np.asarray(j3)
    a = np.zeros_like(j3_arr, dtype=float)

    # Selection rule: m1 + m2 + m3 must be 0
    if (m1 + m2 + m3) != 0:
        warnings.warn("Value of m1+m2+m3 is not zero")
        return a

    # Smallest requested j3 that can be non-zero
    rec_end = max(abs(j1 - j2), abs(m1 + m2), int(np.min(j3_arr)))

    # values index: idx=0 -> j3 = j1+j2; idx=1 -> j1+j2-1; ... down to rec_end
    top = j1 + j2
    if rec_end > top:
        # No valid j3 in the allowed triangular range
        for elem in np.unique(j3_arr):
            warnings.warn(f"Triangular equation for j3 is violated by {elem}")
        return a

    values_len = top + 1 - rec_end
    values = np.zeros(values_len, dtype=float)

    # Initial value at j3 = j1 + j2 (idx 0)
    values[0] = (
        (-1) ** (j1 + j2 + m1 + m2)
        * np.sqrt(
            facvec[2 * j1]
            * facvec[2 * j2]
            * facvec[j1 + j2 - m1 - m2]
            * facvec[j1 + j2 + m1 + m2]
            / (
                facvec[2 * j1 + 2 * j2 + 1]
                * facvec[j1 - m1]
                * facvec[j1 + m1]
                * facvec[j2 - m2]
                * facvec[j2 + m2]
            )
        )
    )

    if rec_end < top:
        # Prepare C(j3) to speed up recursion
        C = np.zeros(top - rec_end, dtype=float)

        # C at j3 = j1 + j2 (for computing the next step)
        C[0] = np.sqrt(
            ((top) ** 2 - (j1 - j2) ** 2)
            * ((top + 1) ** 2 - (top) ** 2)
            * ((top) ** 2 - (m3) ** 2)
        )

        # Second value: j3 = top - 1 (idx 1)
        values[1] = (
            (2 * top + 1)
            * (j1 * (j1 + 1) * m3 - j2 * (j2 + 1) * m3 - top * (top + 1) * (m2 - m1))
            / ((top + 1) * C[0])
            * values[0]
        )

        # Remaining recursion downward
        for idx in range(1, top - rec_end):
            j3_tmp = top - idx  # current j3 corresponding to values[idx]
            # C at current j3_tmp
            C[idx] = np.sqrt(
                (j3_tmp**2 - (j1 - j2) ** 2)
                * ((top + 1) ** 2 - j3_tmp**2)
                * (j3_tmp**2 - m3**2)
            )

            # Recurrence:
            # values[idx+1] corresponds to j3 = j3_tmp - 1
            numerator = (
                -(j3_tmp * C[idx - 1] * values[idx - 1])
                + (2 * j3_tmp + 1)
                * (j1 * (j1 + 1) * m3 - j2 * (j2 + 1) * m3 - j3_tmp * (j3_tmp + 1) * (m2 - m1))
                * values[idx]
            )
            values[idx + 1] = numerator / ((j3_tmp + 1) * C[idx])

    # Map requested j3 values to the precomputed `values`
    # values[0] -> j3 = top; values[-1] -> j3 = rec_end
    for elem in np.unique(j3_arr):
        if (elem <= top) and (rec_end <= elem):
            idx = top - int(elem)  # index into values
            a[j3_arr == elem] = values[idx]
        else:
            warnings.warn(f"Triangular equation for j3 is violated by {elem}")

    return a


In [16]:
import numpy as np
from scipy.linalg import block_diag

def rotate_T(T, alpha, beta, gamma, facvec):
    """
    Rotate the T-matrix using Wigner-D matrices.

    Parameters
    ----------
    T : ndarray, shape (dim, dim, k)
        T-matrices (3D array).
    alpha, beta, gamma : float
        Euler angles.
    facvec : ndarray
        Precomputed factorials, length >= 2*NMax.

    Returns
    -------
    T_new : ndarray
        Rotated T-matrices, same shape as T.
    """
    if alpha == 0 and beta == 0 and gamma == 0:
        return T.copy()

    dim = T.shape[0]
    if T.shape[1] != dim:
        raise ValueError(f"Dimension of T-matrix {T.shape} is not valid.")

    # Recover N from dim = 2 * N * (N + 2)
    N = np.sqrt(1 + dim/2) - 1
    if not N.is_integer():
        raise ValueError(f"Dimension {dim} is not consistent with T-matrix size.")
    N = int(N)

    # Rotation matrix
    D = wignerD_T_rotation(N, alpha, beta, gamma, facvec)

    # Apply rotation for each slice
    T_new = np.zeros_like(T, dtype=complex)
    for idx in range(T.shape[2]):
        T_new[:, :, idx] = D @ T[:, :, idx] @ D.conj().T

    return T_new


def wignerD_T_rotation(NMax, alpha, beta, gamma, facvec):
    """
    Return the rotation matrix to apply to a T-matrix.

    The order is different than the standard order:
    for n = 1..NMax, for m = -n..n
    """
    D_blocks = []
    for n in range(1, NMax+1):
        Dn = wignerD(n, alpha, beta, gamma, facvec)
        D_blocks.append(Dn)

    # blkdiag(D{:}, D{:}) in MATLAB
    D_res = block_diag(*D_blocks, *D_blocks)
    return D_res

from math import sqrt
from scipy.special import lpmv

def transl_coeff(mu, nu, m, n, kr_vec, regular_flag, facvec, wing):
    """
    Compute translation coefficients A and B.

    Parameters
    ----------
    mu, nu, m, n : int
        Angular momentum indices.
    kr_vec : array-like, shape (3,)
        Effective translation vector (x,y,z).
    regular_flag : bool
        If True use regular coefficients (spherical Bessel), else irregular (spherical Hankel).
    facvec : ndarray, shape (171,)
        MATLAB-style factorials: facvec[k] = (k-1)!.
    wing : ndarray, shape (31,15,15)
        Precomputed Wigner-3j related coefficients, MATLAB-style.

    Returns
    -------
    A, B : complex
        Translation coefficients.
    """
    if abs(mu) > nu or abs(m) > n or n <= 0 or nu <= 0:
        raise ValueError("Indices chosen are not possible")

    kr, theta, phi = cts(kr_vec[0], kr_vec[1], kr_vec[2])

    A = 0.0 + 0.0j
    B = 0.0 + 0.0j

    if np.isclose(kr, 0.0):
        if regular_flag and (m == mu and n == nu):
            A = 1.0
        return A, B

    # facvec is shifted: facvec[k] = (k-1)!
    def fval(idx):
        return facvec[idx]  # already MATLAB-style

    pre_common = ((-1)**m) * (1j**(nu-n)) * (2*nu+1)/(2*nu*(nu+1)) \
        * sqrt(fval(n+m+1)*fval(nu-mu+1)/(fval(n-m+1)*fval(nu+mu+1))) \
        * np.exp(1j*(m-mu)*phi)

    pre_norm = sqrt((2*n+1)*fval(n-m+1)/fval(n+m+1)/n/(n+1)) \
             / sqrt((2*nu+1)*fval(nu-mu+1)/fval(nu+mu+1)/nu/(nu+1))

    f = sph_bessel if regular_flag else sph_hankel

    # First loop: p = n+nu down to ...
    for p in range(n+nu, max(abs(n-nu), abs(m-mu))-1, -2):
        order = abs(m-mu)
        leg = lpmv(order, p, np.cos(theta))
        if (m-mu) < 0:
            leg = fval(p-order+1)/fval(p+order+1) * leg * ((-1)**(m-mu))

        w3j = wigner3j(n, nu, p, m, -mu, -m+mu, facvec)

        pre_common_p = (2*p+1) * sqrt(fval(p-m+mu+1)/fval(p+m-mu+1)) \
                     * f(kr, p) * (1j**p) * w3j * leg

        # MATLAB: wing(p+1,n,nu)
        wing_val = wing[p, n-1, nu-1]

        A += wing_val * (n*(n+1) + nu*(nu+1) - p*(p+1)) * pre_common_p

    # Second loop: p = n+nu-1 down to ...
    for p in range(n+nu-1, max(abs(n-nu)+1, abs(m-mu))-1, -2):
        order = abs(m-mu)
        leg = lpmv(order, p, np.cos(theta))
        if (m-mu) < 0:
            leg = fval(p-order+1)/fval(p+order+1) * leg * ((-1)**(m-mu))

        w3j = wigner3j(n, nu, p, m, -mu, -m+mu, facvec)

        pre_common_p = (2*p+1) * sqrt(fval(p-m+mu+1)/fval(p+m-mu+1)) \
                     * f(kr, p) * (1j**p) * w3j * leg

        # MATLAB: wing(p,n,nu)
        wing_val = wing[p-1, n-1, nu-1]

        sqrt_factor = sqrt((n+nu+1+p)*(n+nu+1-p)*(p+n-nu)*(p-n+nu))
        B += wing_val * sqrt_factor * pre_common_p

    A *= pre_norm * pre_common
    B *= pre_norm * pre_common

    return A, B

import numpy as np
from math import sqrt
from .transl_coeff import transl_coeff  # import your earlier function

def T_mat_combine(N, T_mat, positions, kbp, kbm, facvec, wing):
    """
    Calculate the global T-matrix of several local T-matrices at different positions.

    Parameters
    ----------
    N : int
        Order of the T matrices given and the final T matrix
    T_mat : ndarray
        Local T matrices, shape (size_t, size_t, J)
    positions : ndarray
        Positions of the scatterers, shape (J, 3)
    kbp, kbm : float
        Length of the wavevector in the medium
    facvec : ndarray
        Precomputed factorials
    wing : ndarray
        Precomputed Wigner 3j symbols

    Returns
    -------
    T : ndarray
        Global T matrix of the scatterers
    """
    J = positions.shape[0]  # Number of scatterers
    size_t = 2 * N * (N + 2)
    dim = J * size_t

    # Check matrix dimensions
    if T_mat.shape != (size_t, size_t, J):
        raise ValueError("Number of positions not equal to number of given T-matrices.")

    # Build M_inv
    M_inv = np.eye(dim, dtype=complex)
    for ir in range(J):
        for ic in range(J):
            if ir != ic:
                transl_mat = get_transl_mat(
                    positions[ic, :] - positions[ir, :],
                    N,
                    False,
                    kbp,
                    kbm,
                    facvec,
                    wing,
                )
                row_slice = slice(ir * size_t, (ir + 1) * size_t)
                col_slice = slice(ic * size_t, (ic + 1) * size_t)
                M_inv[row_slice, col_slice] = -T_mat[:, :, ir] @ transl_mat

    # Construct block diagonal matrix of T_mat
    T_blocks = [T_mat[:, :, j] for j in range(J)]
    M = np.linalg.solve(M_inv, block_diag(*T_blocks))

    # U transformation
    U = np.zeros((size_t, dim), dtype=complex)
    for ic in range(J):
        U[:, ic * size_t : (ic + 1) * size_t] = get_transl_mat(
            positions[ic, :], N, True, kbp, kbm, facvec, wing
        )

    # V transformation
    V = np.zeros((dim, size_t), dtype=complex)
    for ic in range(J):
        V[ic * size_t : (ic + 1) * size_t, :] = get_transl_mat(
            -positions[ic, :], N, True, kbp, kbm, facvec, wing
        )

    return U @ M @ V


def get_transl_mat(r_vec, N, flag, kbp, kbm, facvec, wing):
    """
    Build translation coefficient matrix.

    Parameters
    ----------
    r_vec : ndarray
        Translation vector (3,)
    N : int
        Maximum order
    flag : bool
        Whether to use regular coefficients
    kbp, kbm : float
        Wavevector lengths
    facvec : ndarray
        Precomputed factorials
    wing : ndarray
        Precomputed Wigner 3j symbols

    Returns
    -------
    res : ndarray
        Translation coefficient matrix
    """
    size_block = N * (N + 2)
    Ap = np.zeros((size_block, size_block), dtype=complex)
    Bp = np.zeros((size_block, size_block), dtype=complex)

    ir = -1
    for n in range(1, N + 1):
        for m in range(-n, n + 1):
            ir += 1
            ic = -1
            for nu in range(1, N + 1):
                for mu in range(-nu, nu + 1):
                    ic += 1
                    Ap[ir, ic], Bp[ir, ic] = transl_coeff(
                        m, n, mu, nu, kbp * r_vec, flag, facvec, wing
                    )

    if kbm == kbp:
        Am = Ap
        Bm = Bp
    else:
        Am = np.zeros((size_block, size_block), dtype=complex)
        Bm = np.zeros((size_block, size_block), dtype=complex)
        ir = -1
        for n in range(1, N + 1):
            for m in range(-n, n + 1):
                ir += 1
                ic = -1
                for nu in range(1, N + 1):
                    for mu in range(-nu, nu + 1):
                        ic += 1
                        Am[ir, ic], Bm[ir, ic] = transl_coeff(
                            m, n, mu, nu, kbm * r_vec, flag, facvec, wing
                        )

    # Construct block matrix
    res = np.block(
        [
            [Ap + Bp, np.zeros((size_block, size_block), dtype=complex)],
            [np.zeros((size_block, size_block), dtype=complex), Am - Bm],
        ]
    )
    return res


def block_diag(*arrs):
    """Replicate MATLAB blkdiag."""
    if len(arrs) == 0:
        return np.zeros((0, 0))

    shapes = np.array([a.shape for a in arrs])
    out = np.zeros(np.sum(shapes, axis=0), dtype=complex)
    r, c = 0, 0
    for a in arrs:
        rr, cc = a.shape
        out[r : r + rr, c : c + cc] = a
        r += rr
        c += cc
    return out



ImportError: attempted relative import with no known parent package

In [ ]:
def T_mat_global(*args):
    """
    Calculate a global T-matrix for multiple T-matrices at different positions and rotations.

    Parameters
    ----------
    *args : sequence
        Arguments passed in order:
        - T_mats (list of dicts) : each dict has field "T" with shape (n_freqs, dim, dim)
        - positions (ndarray)    : shape (J, 3)
        - rotations (ndarray)    : Euler angles, shape (J, 3)
        - epsi (callable/array)  : permittivity as function of freq
        - mu (callable/array)    : permeability as function of freq
        - kappa (callable/array) : chirality parameter as function of freq
        - freqs (ndarray)        : frequency array
        - facvec (ndarray)       : precomputed factorials
        - wing (ndarray)         : precomputed Wigner 3j symbols

    Returns
    -------
    T : ndarray
        Global T-matrix for each frequency, shape (n_freqs, dim, dim)
    N : int
        Multipole expansion order
    """
    c = 299792.458  # speed of light, same units as freqs (THz if consistent)

    # Parse inputs
    positions = args[-7]
    rotations = args[-6]
    epsi      = args[-5]
    mu        = args[-4]
    kappa     = args[-3]
    freqs     = args[-2]
    facvec    = args[-1]
    wing      = args[-0]  # last element

    T_mats = args[:-8]  # all local T-matrices
    nargs = len(args)

    # Determine multipole expansion order from T matrix size
    dim = T_mats[0]["T"].shape[1]
    N = int(np.sqrt(1 + dim / 2) - 1)

    # Allocate result
    T = np.zeros((len(freqs), dim, dim), dtype=complex)

    # Loop over frequencies
    for idx_freq, f in enumerate(freqs):
        T_tmp = np.zeros((dim, dim, nargs - 8), dtype=complex)

        # Rotate each local T-matrix
        for idx_T in range(nargs - 8):
            T_local = T_mats[idx_T]["T"][idx_freq, :, :]
            alpha, beta, gamma = rotations[idx_T, :]
            T_tmp[:, :, idx_T] = rotate_T(T_local, alpha, beta, gamma, facvec)

        # Compute medium-dependent wave numbers
        kbp = (
            2
            * np.pi
            * (np.sqrt(epsi(f) * mu(f)) + kappa(f))
            * f
            / c
        )
        kbm = (
            2
            * np.pi
            * (np.sqrt(epsi(f) * mu(f)) - kappa(f))
            * f
            / c
        )

        # Combine into global T
        T[idx_freq, :, :] = T_mat_combine(N, T_tmp, positions, kbp, kbm, facvec, wing)

    return T, N